# Camada Gold — Agregações e Visões Analíticas (`03_agregacoes_gold.py`)

Este notebook constrói os **Data Marts** da Camada Gold a partir das tabelas refinadas da Camada Silver (`copa_do_mundo.silver`).

### Tabelas Analíticas Geradas:
1. **`gold_squad_metrics`**: Visão executiva por seleção (média de idade, valor de mercado acumulado/médio, total de convocados e distribuição por ligas).
2. **`gold_player_performance`**: Métricas agregadas por jogador (gols, assistências, minutos jogados e eficiência de finalização) combinadas com idade atual.
3. **`gold_referee_analytics`**: Perfil de arbitragem (média de faltas por jogo, cartões e distribuição por confederação).
4. **`gold_match_summary`**: Dados de partidas consolidado para análise de desempenho por fase e predições.

In [0]:
from pyspark.sql import functions as F

CATALOG_NAME = "copa_do_mundo"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

# Criação do schema Gold no Unity Catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{GOLD_SCHEMA}")

# Leitura das tabelas Silver refinadas
df_squads = spark.read.table(
    f"{CATALOG_NAME}.{SILVER_SCHEMA}.squads_and_players_silver"
)
df_player_stats = spark.read.table(
    f"{CATALOG_NAME}.{SILVER_SCHEMA}.player_stats_silver"
)
df_referees = spark.read.table(
    f"{CATALOG_NAME}.{SILVER_SCHEMA}.referees_silver"
)
df_matches = spark.read.table(
    f"{CATALOG_NAME}.{SILVER_SCHEMA}.matches_detailed_silver"
)
df_predictions = spark.read.table(
    f"{CATALOG_NAME}.{SILVER_SCHEMA}.match_prediction_features_silver"
)

## 1. Análise de Seleções (`gold_squad_metrics`)
Agregação por seleção trazendo métricas de idade e financeiras (idade média, total de valor de mercado em EUR, total de atletas convocados).

In [0]:
# Cálculo da idade do atleta com base na data de nascimento (considerando 2026)
df_squads_with_age = df_squads.withColumn(
    "idade", F.floor(F.datediff(F.lit("2026-06-11"), F.col("date_of_birth")) / 365.25)
)

gold_squad_metrics = df_squads_with_age.groupBy("team_id").agg(
    F.count("player_id").alias("total_convocados"),
    F.round(F.avg("idade"), 1).alias("idade_media"),
    F.round(F.sum("market_value_eur"), 2).alias("valor_total_mercado_eur"),
    F.round(F.avg("market_value_eur"), 2).alias("valor_medio_jogador_eur"),
    F.countDistinct("club_team").alias("total_clubes_representados"),
)

display(gold_squad_metrics.limit(10))

## 2. Desempenho de Jogadores (`gold_player_performance`)
Consolidação das estatísticas individuais dos atletas com dados cadastrais e idade calculada.

In [0]:
gold_player_performance = (
    df_squads_with_age.select(
        "player_id",
        "player_name",
        "team_id",
        "position",
        "idade",
        "club_team",
        "market_value_eur",
        "date_of_birth",
    )
    .join(
        df_player_stats.drop("player_name", "team_id", "position"),
        on=["player_id", "date_of_birth"],
        how="inner",
    )
    .select(
        "player_id",
        "player_name",
        "team_id",
        "position",
        "idade",
        "club_team",
        "market_value_eur",
        "minutes_played",
        "goals",
        "assists",
        "shots",
        "shots_on_target",
    )
)

display(gold_player_performance.limit(10))

## 3. Análise da Arbitragem (`gold_referee_analytics`)
Métricas detalhadas de atuação dos árbitros, incluindo média de faltas por jogo apitado.

In [0]:
gold_referee_analytics = df_referees.withColumn(
    "media_faltas_por_jogo",
    F.when(
        F.col("total_jogos_apitados") > 0,
        F.round(
            F.col("total_faltas_marcadas") / F.col("total_jogos_apitados"), 2
        ),
    ).otherwise(0.0),
).select(
    "referee_id",
    F.col("name").alias("referee_name"),
    "country",
    "confederacao",
    "total_jogos_apitados",
    "total_faltas_marcadas",
    "media_faltas_por_jogo",
)

display(gold_referee_analytics.limit(10))

## 4. Salvamento em Delta Lake e Disponibilização no Unity Catalog

In [0]:
gold_tables = {
    "gold_squad_metrics": gold_squad_metrics,
    "gold_player_performance": gold_player_performance,
    "gold_referee_analytics": gold_referee_analytics,
}

for table_name, df_gold in gold_tables.items():
    full_path = f"{CATALOG_NAME}.{GOLD_SCHEMA}.{table_name}"
    print(f"Escrevendo tabela Gold: {full_path}...")

    df_gold.write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).saveAsTable(full_path)

print("\n🚀 Camada Gold gerada e persistida com sucesso no Unity Catalog!")